# PPTX Agent (All-in-One) for Google Colab

This notebook is self-contained: all agent code is inside this notebook.

Structure:
1. Libraries
2. API key loading
3. Full agent code
4. Run block (set template path + prompt)

In [ ]:
# 1) Libraries (install + imports)
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(['apt-get', '-qq', 'update'], check=False)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libreoffice', 'poppler-utils'], check=False)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pydantic>=2.8',
    'pydantic-ai-slim[groq]>=0.0.24',
    'python-pptx>=1.0.2',
    'google-generativeai>=0.7.2',
    'markitdown[pptx]>=0.0.1a3',
    'Pillow>=10.4.0',
    'defusedxml>=0.7.1'
], check=False)

import json
import shutil
import tempfile
import zipfile
import subprocess as sp
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Any

import defusedxml.minidom
import google.generativeai as genai
from PIL import Image
from pydantic import BaseModel, Field, model_validator
from pydantic_ai import Agent, ModelSettings
from pydantic_ai.models.groq import GroqModel
from pydantic_ai.providers.groq import GroqProvider
from pptx import Presentation
from pptx.enum.shapes import PP_PLACEHOLDER

print('Libraries ready.')

In [ ]:
# 2) API keys from Colab Secrets (and env defaults)
try:
    from google.colab import userdata
    api_key_from_secrets = userdata.get('GROQ_API_KEY')
    if api_key_from_secrets:
        os.environ['GROQ_API_KEY'] = api_key_from_secrets
        print(f'GROQ key from Colab Secrets: {api_key_from_secrets[:5]}...{api_key_from_secrets[-5:]}')
    else:
        print('GROQ_API_KEY not found in Colab secrets.')
except Exception as e:
    print(f'Could not retrieve GROQ_API_KEY from Colab secrets: {e}')

try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        os.environ['GEMINI_API_KEY'] = gemini_key
        print(f'GEMINI key from Colab Secrets: {gemini_key[:5]}...{gemini_key[-5:]}')
    else:
        print('GEMINI_API_KEY not found in Colab secrets (visual QA will be skipped).')
except Exception as e:
    print(f'Could not retrieve GEMINI_API_KEY from Colab secrets: {e}')

os.environ.setdefault('GROQ_MODEL', 'gpt-oss-120b')
os.environ.setdefault('GEMINI_VISION_MODEL', 'gemini-2.0-flash')
os.environ.setdefault('PPTX_AGENT_WORKDIR', '/content/.pptx_agent_work')
os.environ.setdefault('PPTX_AGENT_MAX_FIX_LOOPS', '2')
os.environ.setdefault('PPTX_AGENT_DEFAULT_LANGUAGE', 'fr')
os.environ.setdefault('PPTX_AGENT_ENABLE_VISUAL_QA', 'true')

print('GROQ key set:', bool(os.getenv('GROQ_API_KEY')))
print('GEMINI key set:', bool(os.getenv('GEMINI_API_KEY')))

In [ ]:
# 3) Full agent code (all-in-one)

def _env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'on'}


def _env_int(name: str, default: int) -> int:
    raw = os.getenv(name)
    if raw is None:
        return default
    try:
        return int(raw)
    except ValueError:
        return default


@dataclass(slots=True)
class AgentSettings:
    groq_api_key: str | None
    groq_model: str
    gemini_api_key: str | None
    gemini_vision_model: str
    workdir: Path
    max_fix_loops: int
    default_language: str
    enable_visual_qa: bool


def load_settings() -> AgentSettings:
    return AgentSettings(
        groq_api_key=os.getenv('GROQ_API_KEY'),
        groq_model=os.getenv('GROQ_MODEL', 'gpt-oss-120b'),
        gemini_api_key=os.getenv('GEMINI_API_KEY'),
        gemini_vision_model=os.getenv('GEMINI_VISION_MODEL', 'gemini-2.0-flash'),
        workdir=Path(os.getenv('PPTX_AGENT_WORKDIR', '/content/.pptx_agent_work')).resolve(),
        max_fix_loops=max(1, _env_int('PPTX_AGENT_MAX_FIX_LOOPS', 2)),
        default_language=os.getenv('PPTX_AGENT_DEFAULT_LANGUAGE', 'fr').strip().lower(),
        enable_visual_qa=_env_bool('PPTX_AGENT_ENABLE_VISUAL_QA', True),
    )


class SlideSummary(BaseModel):
    slide_index: int = Field(ge=1)
    layout_name: str | None = None
    placeholder_names: list[str] = Field(default_factory=list)
    text_preview: str = ''


class DeckAnalysis(BaseModel):
    source_path: str
    slide_count: int = Field(ge=0)
    detected_language: str = 'fr'
    extracted_text: str = ''
    thumbnail_paths: list[str] = Field(default_factory=list)
    slides: list[SlideSummary] = Field(default_factory=list)


class StructureOperationType(str, Enum):
    delete_slide = 'delete_slide'
    duplicate_slide = 'duplicate_slide'
    add_layout_slide = 'add_layout_slide'
    reorder_slides = 'reorder_slides'


class StructureOperation(BaseModel):
    op: StructureOperationType
    slide_index: int | None = Field(default=None, ge=1)
    target_index: int | None = Field(default=None, ge=1)
    layout_index: int | None = Field(default=None, ge=1)
    new_order: list[int] = Field(default_factory=list)
    reason: str = ''

    @model_validator(mode='after')
    def validate_by_op(self):
        if self.op == StructureOperationType.delete_slide and self.slide_index is None:
            raise ValueError('delete_slide requires slide_index')
        if self.op == StructureOperationType.duplicate_slide and self.slide_index is None:
            raise ValueError('duplicate_slide requires slide_index')
        if self.op == StructureOperationType.add_layout_slide and self.layout_index is None:
            raise ValueError('add_layout_slide requires layout_index')
        if self.op == StructureOperationType.reorder_slides and not self.new_order:
            raise ValueError('reorder_slides requires new_order')
        return self


class StructurePlan(BaseModel):
    rationale: str = ''
    operations: list[StructureOperation] = Field(default_factory=list)


class SlideContentUpdate(BaseModel):
    slide_index: int = Field(ge=1)
    title: str | None = None
    subtitle: str | None = None
    bullets: list[str] = Field(default_factory=list)
    body_paragraphs: list[str] = Field(default_factory=list)
    notes: str | None = None
    remove_empty_placeholders: bool = True


class ContentPlan(BaseModel):
    language: str = 'fr'
    slides: list[SlideContentUpdate] = Field(default_factory=list)


class QAIssueSeverity(str, Enum):
    low = 'low'
    medium = 'medium'
    high = 'high'


class QAIssueCategory(str, Enum):
    placeholder = 'placeholder'
    typo = 'typo'
    overflow = 'overflow'
    overlap = 'overlap'
    alignment = 'alignment'
    contrast = 'contrast'
    spacing = 'spacing'
    other = 'other'


class QAIssue(BaseModel):
    slide_index: int = Field(ge=1)
    severity: QAIssueSeverity
    category: QAIssueCategory
    description: str
    fix_hint: str = ''


class QAReport(BaseModel):
    mode: str
    issues: list[QAIssue] = Field(default_factory=list)
    notes: list[str] = Field(default_factory=list)


class RunArtifacts(BaseModel):
    workdir: str
    unpacked_dir: str
    analysis_text_path: str | None = None
    rendered_image_paths: list[str] = Field(default_factory=list)
    report_json_path: str | None = None


class RunReport(BaseModel):
    input_pptx: str
    output_pptx: str
    instruction: str
    structure_plan: StructurePlan
    content_plan: ContentPlan
    qa_reports: list[QAReport]
    final_issue_count: int
    artifacts: RunArtifacts


SMART_QUOTE_REPLACEMENTS = {
    '“': '&#x201C;',
    '”': '&#x201D;',
    '‘': '&#x2018;',
    '’': '&#x2019;',
}


def command_exists(command: str) -> bool:
    return shutil.which(command) is not None


def run_command(command: list[str], cwd: Path | None = None) -> sp.CompletedProcess:
    return sp.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=True, check=False)


def extract_json_object(raw_text: str) -> dict[str, Any]:
    raw = (raw_text or '').strip()
    if not raw:
        return {}

    if raw.startswith('```'):
        raw = raw.strip('`').strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    start = raw.find('{')
    end = raw.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return {}

    candidate = raw[start:end + 1]
    try:
        return json.loads(candidate)
    except Exception:
        return {}


def natural_sort_key(path: Path) -> tuple:
    key: list[Any] = []
    token = ''
    mode_digit = None
    for ch in path.name:
        is_digit = ch.isdigit()
        if mode_digit is None:
            mode_digit = is_digit
            token = ch
            continue
        if is_digit == mode_digit:
            token += ch
            continue
        key.append(int(token) if mode_digit else token.lower())
        token = ch
        mode_digit = is_digit
    if token:
        key.append(int(token) if mode_digit else token.lower())
    return tuple(key)


def _extract_text_markitdown(pptx_path: Path) -> str:
    result = run_command([sys.executable, '-m', 'markitdown', str(pptx_path)])
    if result.returncode != 0:
        return ''
    return result.stdout.strip()


def _extract_text_python_pptx(pptx_path: Path) -> str:
    prs = Presentation(str(pptx_path))
    lines: list[str] = []
    for index, slide in enumerate(prs.slides, start=1):
        lines.append(f'# Slide {index}')
        for shape in slide.shapes:
            if shape.has_text_frame and shape.text.strip():
                lines.append(shape.text.strip())
        lines.append('')
    return '\n'.join(lines).strip()


def detect_language(text: str, default: str = 'fr') -> str:
    if not text:
        return default
    lowered = text.lower()
    french_markers = [' le ', ' la ', ' les ', ' de ', ' pour ', ' avec ']
    english_markers = [' the ', ' and ', ' for ', ' with ', ' is ', ' are ']
    fr_score = sum(marker in lowered for marker in french_markers)
    en_score = sum(marker in lowered for marker in english_markers)
    return 'en' if en_score > fr_score else 'fr'


def _shape_text_preview(text: str, max_len: int = 180) -> str:
    cleaned = ' '.join(text.split())
    if len(cleaned) <= max_len:
        return cleaned
    return cleaned[:max_len - 1].rstrip() + '...'


def _collect_slide_summary(pptx_path: Path) -> list[SlideSummary]:
    prs = Presentation(str(pptx_path))
    slides: list[SlideSummary] = []
    for index, slide in enumerate(prs.slides, start=1):
        placeholders: list[str] = []
        text_blocks: list[str] = []
        for shape in slide.shapes:
            if getattr(shape, 'is_placeholder', False):
                placeholders.append(shape.name)
            if shape.has_text_frame and shape.text.strip():
                text_blocks.append(shape.text.strip())

        layout_name = slide.slide_layout.name if slide.slide_layout is not None else None
        slides.append(SlideSummary(
            slide_index=index,
            layout_name=layout_name,
            placeholder_names=placeholders,
            text_preview=_shape_text_preview(' | '.join(text_blocks)),
        ))
    return slides


def _resolve_soffice_command() -> str | None:
    for candidate in ('soffice', 'libreoffice'):
        if command_exists(candidate):
            return candidate
    return None


def render_slides_to_images(pptx_path: Path, output_dir: Path, prefix: str = 'slide') -> list[Path]:
    soffice_cmd = _resolve_soffice_command()
    if soffice_cmd is None or not command_exists('pdftoppm'):
        raise RuntimeError('soffice and pdftoppm are required for image rendering')

    output_dir.mkdir(parents=True, exist_ok=True)
    convert = run_command([soffice_cmd, '--headless', '--convert-to', 'pdf', '--outdir', str(output_dir), str(pptx_path)])
    if convert.returncode != 0:
        raise RuntimeError(f'PDF conversion failed: {convert.stderr or convert.stdout}')

    pdf_path = output_dir / f'{pptx_path.stem}.pdf'
    if not pdf_path.exists():
        candidates = sorted(output_dir.glob('*.pdf'))
        if not candidates:
            raise RuntimeError('No PDF generated by soffice')
        pdf_path = candidates[0]

    render = run_command(['pdftoppm', '-png', '-r', '150', str(pdf_path), str(output_dir / prefix)])
    if render.returncode != 0:
        raise RuntimeError(f'PNG render failed: {render.stderr or render.stdout}')

    images = sorted(output_dir.glob(f'{prefix}-*.png'), key=natural_sort_key)
    if not images:
        images = sorted(output_dir.glob(f'{prefix}*.png'), key=natural_sort_key)
    if not images:
        raise RuntimeError('No slide images generated')
    return images


def analyze_template(pptx_path: Path, workdir: Path, default_language: str = 'fr') -> DeckAnalysis:
    workdir.mkdir(parents=True, exist_ok=True)
    thumbs_dir = workdir / 'thumbnails'
    thumbs_dir.mkdir(parents=True, exist_ok=True)

    extracted = _extract_text_markitdown(pptx_path)
    if not extracted:
        extracted = _extract_text_python_pptx(pptx_path)

    slides = _collect_slide_summary(pptx_path)
    thumbnail_paths: list[str] = []
    notes: list[str] = []
    try:
        images = render_slides_to_images(pptx_path, thumbs_dir, prefix='analysis-slide')
        thumbnail_paths = [str(p) for p in images]
    except Exception as exc:
        notes.append(str(exc))

    language = detect_language(extracted, default=default_language)

    payload = extracted
    if notes:
        payload += '\n\n[notes]\n' + '\n'.join(notes)
    (workdir / 'analysis.txt').write_text(payload, encoding='utf-8')

    return DeckAnalysis(
        source_path=str(pptx_path),
        slide_count=len(slides),
        detected_language=language,
        extracted_text=extracted,
        thumbnail_paths=thumbnail_paths,
        slides=slides,
    )


def _pretty_print_xml(xml_file: Path) -> None:
    try:
        content = xml_file.read_text(encoding='utf-8')
        dom = defusedxml.minidom.parseString(content)
        xml_file.write_bytes(dom.toprettyxml(indent='  ', encoding='utf-8'))
    except Exception:
        return


def _escape_smart_quotes(xml_file: Path) -> None:
    try:
        content = xml_file.read_text(encoding='utf-8')
        for char, entity in SMART_QUOTE_REPLACEMENTS.items():
            content = content.replace(char, entity)
        xml_file.write_text(content, encoding='utf-8')
    except Exception:
        return


def unpack_pptx(input_pptx: Path, output_dir: Path) -> None:
    if not input_pptx.exists():
        raise FileNotFoundError(f'Input PPTX not found: {input_pptx}')
    output_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(input_pptx, 'r') as archive:
        archive.extractall(output_dir)

    for path in list(output_dir.rglob('*.xml')) + list(output_dir.rglob('*.rels')):
        _pretty_print_xml(path)
        _escape_smart_quotes(path)


def _load_dom(path: Path):
    return defusedxml.minidom.parse(str(path))


def _save_dom(path: Path, dom) -> None:
    path.write_bytes(dom.toxml(encoding='utf-8'))


def _presentation_paths(unpacked_dir: Path) -> tuple[Path, Path]:
    pres = unpacked_dir / 'ppt' / 'presentation.xml'
    rels = unpacked_dir / 'ppt' / '_rels' / 'presentation.xml.rels'
    if not pres.exists() or not rels.exists():
        raise FileNotFoundError('presentation.xml or presentation.xml.rels missing')
    return pres, rels


def _slide_id_nodes(unpacked_dir: Path):
    pres_path, _ = _presentation_paths(unpacked_dir)
    dom = _load_dom(pres_path)
    sld_id_lst_nodes = dom.getElementsByTagName('p:sldIdLst')
    if not sld_id_lst_nodes:
        raise RuntimeError('<p:sldIdLst> not found')
    sld_id_lst = sld_id_lst_nodes[0]
    nodes = [n for n in sld_id_lst.childNodes if n.nodeType == n.ELEMENT_NODE and n.tagName == 'p:sldId']
    return dom, sld_id_lst, nodes


def _rid_to_slide_target(unpacked_dir: Path) -> dict[str, str]:
    _, rels_path = _presentation_paths(unpacked_dir)
    rels_dom = _load_dom(rels_path)
    mapping: dict[str, str] = {}
    for rel in rels_dom.getElementsByTagName('Relationship'):
        rel_type = rel.getAttribute('Type')
        target = rel.getAttribute('Target')
        if rel_type.endswith('/slide') and target.startswith('slides/'):
            mapping[rel.getAttribute('Id')] = target
    return mapping


def list_slide_sequence(unpacked_dir: Path) -> list[str]:
    _, _, slide_nodes = _slide_id_nodes(unpacked_dir)
    mapping = _rid_to_slide_target(unpacked_dir)
    out: list[str] = []
    for node in slide_nodes:
        rid = node.getAttribute('r:id')
        target = mapping.get(rid)
        if target:
            out.append(Path(target).name)
    return out


def _parse_slide_number(name: str) -> int | None:
    if not name.startswith('slide') or not name.endswith('.xml'):
        return None
    middle = name.replace('slide', '').replace('.xml', '')
    return int(middle) if middle.isdigit() else None


def _next_slide_number(slides_dir: Path) -> int:
    numbers: list[int] = []
    for path in slides_dir.glob('slide*.xml'):
        num = _parse_slide_number(path.name)
        if num is not None:
            numbers.append(num)
    return max(numbers) + 1 if numbers else 1


def _extract_rid_numbers(text: str) -> list[int]:
    numbers: list[int] = []
    marker = 'Id="rId'
    start = 0
    while True:
        idx = text.find(marker, start)
        if idx == -1:
            break
        j = idx + len(marker)
        token = ''
        while j < len(text) and text[j].isdigit():
            token += text[j]
            j += 1
        if token:
            numbers.append(int(token))
        start = j
    return numbers


def _next_rid(rels_path: Path) -> str:
    content = rels_path.read_text(encoding='utf-8')
    ids = _extract_rid_numbers(content)
    return f'rId{max(ids) + 1 if ids else 1}'


def _next_slide_id(unpacked_dir: Path) -> int:
    _, _, slide_nodes = _slide_id_nodes(unpacked_dir)
    ids: list[int] = []
    for node in slide_nodes:
        value = node.getAttribute('id')
        if value.isdigit():
            ids.append(int(value))
    return max(ids) + 1 if ids else 256


def _add_slide_override_content_type(unpacked_dir: Path, slide_name: str) -> None:
    content_types_path = unpacked_dir / '[Content_Types].xml'
    dom = _load_dom(content_types_path)
    root = dom.documentElement
    part_name = f'/ppt/slides/{slide_name}'

    for override in root.getElementsByTagName('Override'):
        if override.getAttribute('PartName') == part_name:
            _save_dom(content_types_path, dom)
            return

    override = dom.createElement('Override')
    override.setAttribute('PartName', part_name)
    override.setAttribute('ContentType', 'application/vnd.openxmlformats-officedocument.presentationml.slide+xml')
    root.appendChild(override)
    _save_dom(content_types_path, dom)


def _add_presentation_relationship(unpacked_dir: Path, slide_name: str) -> str:
    _, rels_path = _presentation_paths(unpacked_dir)
    dom = _load_dom(rels_path)
    root = dom.documentElement
    rid = _next_rid(rels_path)

    rel = dom.createElement('Relationship')
    rel.setAttribute('Id', rid)
    rel.setAttribute('Type', 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/slide')
    rel.setAttribute('Target', f'slides/{slide_name}')
    root.appendChild(rel)
    _save_dom(rels_path, dom)
    return rid


def _insert_slide_id(unpacked_dir: Path, rid: str, target_index: int | None = None) -> None:
    dom, sld_id_lst, slide_nodes = _slide_id_nodes(unpacked_dir)
    node = dom.createElement('p:sldId')
    node.setAttribute('id', str(_next_slide_id(unpacked_dir)))
    node.setAttribute('r:id', rid)

    if target_index is None:
        sld_id_lst.appendChild(node)
    else:
        insert_at = max(1, target_index)
        if insert_at > len(slide_nodes):
            sld_id_lst.appendChild(node)
        else:
            sld_id_lst.insertBefore(node, slide_nodes[insert_at - 1])

    pres_path, _ = _presentation_paths(unpacked_dir)
    _save_dom(pres_path, dom)


def delete_slide(unpacked_dir: Path, slide_index: int) -> None:
    dom, sld_id_lst, slide_nodes = _slide_id_nodes(unpacked_dir)
    if slide_index < 1 or slide_index > len(slide_nodes):
        raise IndexError(f'invalid slide_index: {slide_index}')
    node = slide_nodes[slide_index - 1]
    sld_id_lst.removeChild(node)
    pres_path, _ = _presentation_paths(unpacked_dir)
    _save_dom(pres_path, dom)


def reorder_slides(unpacked_dir: Path, new_order: list[int]) -> None:
    dom, sld_id_lst, slide_nodes = _slide_id_nodes(unpacked_dir)
    count = len(slide_nodes)
    expected = list(range(1, count + 1))
    if sorted(new_order) != expected:
        raise ValueError(f'new_order must be a permutation of {expected}')

    for node in list(slide_nodes):
        sld_id_lst.removeChild(node)
    for idx in new_order:
        sld_id_lst.appendChild(slide_nodes[idx - 1])

    pres_path, _ = _presentation_paths(unpacked_dir)
    _save_dom(pres_path, dom)


def duplicate_slide(unpacked_dir: Path, slide_index: int, target_index: int | None = None) -> str:
    sequence = list_slide_sequence(unpacked_dir)
    if slide_index < 1 or slide_index > len(sequence):
        raise IndexError(f'invalid slide_index: {slide_index}')

    source_name = sequence[slide_index - 1]
    slides_dir = unpacked_dir / 'ppt' / 'slides'
    rels_dir = slides_dir / '_rels'
    rels_dir.mkdir(parents=True, exist_ok=True)

    dest_name = f'slide{_next_slide_number(slides_dir)}.xml'
    shutil.copy2(slides_dir / source_name, slides_dir / dest_name)

    source_rels = rels_dir / f'{source_name}.rels'
    dest_rels = rels_dir / f'{dest_name}.rels'
    if source_rels.exists():
        shutil.copy2(source_rels, dest_rels)

    _add_slide_override_content_type(unpacked_dir, dest_name)
    rid = _add_presentation_relationship(unpacked_dir, dest_name)
    insertion_position = target_index if target_index is not None else slide_index + 1
    _insert_slide_id(unpacked_dir, rid, target_index=insertion_position)
    return dest_name


def add_slide_from_layout(unpacked_dir: Path, layout_index: int, target_index: int | None = None) -> str:
    layout_file = f'slideLayout{layout_index}.xml'
    layout_path = unpacked_dir / 'ppt' / 'slideLayouts' / layout_file
    if not layout_path.exists():
        raise FileNotFoundError(f'layout not found: {layout_file}')

    slides_dir = unpacked_dir / 'ppt' / 'slides'
    rels_dir = slides_dir / '_rels'
    rels_dir.mkdir(parents=True, exist_ok=True)

    slide_name = f'slide{_next_slide_number(slides_dir)}.xml'
    slide_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\n'
    slide_xml += '<p:sld xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" xmlns:p="http://schemas.openxmlformats.org/presentationml/2006/main">\n'
    slide_xml += '  <p:cSld><p:spTree><p:nvGrpSpPr><p:cNvPr id="1" name=""/><p:cNvGrpSpPr/><p:nvPr/></p:nvGrpSpPr><p:grpSpPr><a:xfrm><a:off x="0" y="0"/><a:ext cx="0" cy="0"/><a:chOff x="0" y="0"/><a:chExt cx="0" cy="0"/></a:xfrm></p:grpSpPr></p:spTree></p:cSld>\n'
    slide_xml += '  <p:clrMapOvr><a:masterClrMapping/></p:clrMapOvr>\n'
    slide_xml += '</p:sld>\n'
    (slides_dir / slide_name).write_text(slide_xml, encoding='utf-8')

    rels_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\n'
    rels_xml += '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">\n'
    rels_xml += f'  <Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/slideLayout" Target="../slideLayouts/{layout_file}"/>\n'
    rels_xml += '</Relationships>\n'
    (rels_dir / f'{slide_name}.rels').write_text(rels_xml, encoding='utf-8')

    _add_slide_override_content_type(unpacked_dir, slide_name)
    rid = _add_presentation_relationship(unpacked_dir, slide_name)
    _insert_slide_id(unpacked_dir, rid, target_index=target_index)
    return slide_name


def clean_unreferenced_files(unpacked_dir: Path) -> list[str]:
    removed: list[str] = []
    sequence = set(list_slide_sequence(unpacked_dir))
    slides_dir = unpacked_dir / 'ppt' / 'slides'
    rels_dir = slides_dir / '_rels'

    for slide_file in slides_dir.glob('slide*.xml'):
        if slide_file.name in sequence:
            continue
        slide_file.unlink(missing_ok=True)
        removed.append(str(slide_file.relative_to(unpacked_dir)).replace('\\', '/'))
        rels_file = rels_dir / f'{slide_file.name}.rels'
        if rels_file.exists():
            rels_file.unlink()
            removed.append(str(rels_file.relative_to(unpacked_dir)).replace('\\', '/'))

    trash_dir = unpacked_dir / '[trash]'
    if trash_dir.exists() and trash_dir.is_dir():
        for path in trash_dir.glob('*'):
            if path.is_file():
                path.unlink()
                removed.append(str(path.relative_to(unpacked_dir)).replace('\\', '/'))
        trash_dir.rmdir()

    return removed


def apply_structure_plan(unpacked_dir: Path, plan: StructurePlan) -> list[str]:
    logs: list[str] = []
    for step, operation in enumerate(plan.operations, start=1):
        if operation.op == StructureOperationType.delete_slide:
            delete_slide(unpacked_dir, operation.slide_index or 1)
            logs.append(f'[{step}] delete_slide {operation.slide_index}')
        elif operation.op == StructureOperationType.duplicate_slide:
            created = duplicate_slide(unpacked_dir, operation.slide_index or 1, operation.target_index)
            logs.append(f'[{step}] duplicate_slide -> {created}')
        elif operation.op == StructureOperationType.add_layout_slide:
            created = add_slide_from_layout(unpacked_dir, operation.layout_index or 1, operation.target_index)
            logs.append(f'[{step}] add_layout_slide -> {created}')
        elif operation.op == StructureOperationType.reorder_slides:
            reorder_slides(unpacked_dir, operation.new_order)
            logs.append(f'[{step}] reorder_slides')

    removed = clean_unreferenced_files(unpacked_dir)
    if removed:
        logs.append(f'clean: removed {len(removed)} files')
    return logs


def _condense_xml(xml_file: Path) -> None:
    dom = _load_dom(xml_file)
    for element in dom.getElementsByTagName('*'):
        if element.tagName.endswith(':t'):
            continue
        for child in list(element.childNodes):
            if child.nodeType == child.TEXT_NODE and child.nodeValue and child.nodeValue.strip() == '':
                element.removeChild(child)
    xml_file.write_bytes(dom.toxml(encoding='utf-8'))


def pack_pptx(input_directory: Path, output_file: Path) -> None:
    if not input_directory.is_dir():
        raise NotADirectoryError(f'Not a directory: {input_directory}')

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_root = Path(temp_dir) / 'content'
        shutil.copytree(input_directory, temp_root)

        for pattern in ('*.xml', '*.rels'):
            for xml_file in temp_root.rglob(pattern):
                _condense_xml(xml_file)

        with zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED) as archive:
            for file_path in temp_root.rglob('*'):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(temp_root))


def _trim_line(text: str, max_len: int = 220) -> str:
    cleaned = ' '.join((text or '').split())
    if len(cleaned) <= max_len:
        return cleaned
    return cleaned[:max_len - 1].rstrip() + '...'


def _find_title_shape(slide):
    for shape in slide.shapes:
        if not getattr(shape, 'is_placeholder', False):
            continue
        try:
            ph_type = shape.placeholder_format.type
        except Exception:
            continue
        if ph_type in {PP_PLACEHOLDER.TITLE, PP_PLACEHOLDER.CENTER_TITLE}:
            return shape
    for shape in slide.shapes:
        if shape.has_text_frame:
            return shape
    return None


def _find_subtitle_shape(slide):
    for shape in slide.shapes:
        if not getattr(shape, 'is_placeholder', False):
            continue
        try:
            ph_type = shape.placeholder_format.type
        except Exception:
            continue
        if ph_type == PP_PLACEHOLDER.SUBTITLE:
            return shape
    return None


def _body_text_shapes(slide):
    out = []
    for shape in slide.shapes:
        if not shape.has_text_frame:
            continue
        if getattr(shape, 'is_placeholder', False):
            try:
                ph_type = shape.placeholder_format.type
            except Exception:
                ph_type = None
            if ph_type in {PP_PLACEHOLDER.TITLE, PP_PLACEHOLDER.CENTER_TITLE, PP_PLACEHOLDER.SUBTITLE}:
                continue
        out.append(shape)
    return out


def _write_line(paragraph, text: str, bold_all: bool = False) -> None:
    paragraph.clear()
    cleaned = _trim_line(text)
    if not cleaned:
        return

    if bold_all:
        run = paragraph.add_run()
        run.text = cleaned
        run.font.bold = True
        return

    if ':' in cleaned and cleaned.index(':') < 80:
        left, right = cleaned.split(':', 1)
        label_run = paragraph.add_run()
        label_run.text = left + ':'
        label_run.font.bold = True
        if right.strip():
            value_run = paragraph.add_run()
            value_run.text = ' ' + right.strip()
        return

    run = paragraph.add_run()
    run.text = cleaned


def _write_text_frame(shape, lines: list[str]) -> None:
    if not shape.has_text_frame:
        return
    frame = shape.text_frame
    frame.clear()
    for idx, line in enumerate(lines):
        paragraph = frame.paragraphs[0] if idx == 0 else frame.add_paragraph()
        paragraph.level = 0
        _write_line(paragraph, line, bold_all=False)


def _bold_all_runs(shape) -> None:
    if not shape.has_text_frame:
        return
    for paragraph in shape.text_frame.paragraphs:
        if not paragraph.runs and paragraph.text.strip():
            run = paragraph.add_run()
            run.text = paragraph.text
        for run in paragraph.runs:
            run.font.bold = True


def _remove_empty_placeholders(slide) -> int:
    to_remove = []
    for shape in slide.shapes:
        if not getattr(shape, 'is_placeholder', False):
            continue
        if not shape.has_text_frame:
            continue
        if shape.text.strip():
            continue
        try:
            ph_type = shape.placeholder_format.type
        except Exception:
            continue
        if ph_type in {PP_PLACEHOLDER.TITLE, PP_PLACEHOLDER.CENTER_TITLE}:
            continue
        to_remove.append(shape)

    for shape in to_remove:
        element = shape.element
        parent = element.getparent()
        if parent is not None:
            parent.remove(element)
    return len(to_remove)


def apply_slide_update(slide, update: SlideContentUpdate) -> dict[str, int]:
    counters = {'removed_placeholders': 0}

    title_shape = _find_title_shape(slide)
    if update.title and title_shape is not None:
        title_shape.text = _trim_line(update.title, max_len=180)
        _bold_all_runs(title_shape)

    subtitle_shape = _find_subtitle_shape(slide)
    if update.subtitle and subtitle_shape is not None:
        subtitle_shape.text = _trim_line(update.subtitle, max_len=220)

    body_shapes = _body_text_shapes(slide)
    cursor = 0

    if update.bullets and body_shapes:
        _write_text_frame(body_shapes[cursor], update.bullets[:10])
        cursor += 1

    if update.body_paragraphs and body_shapes:
        target = body_shapes[cursor] if cursor < len(body_shapes) else body_shapes[-1]
        _write_text_frame(target, update.body_paragraphs[:8])

    if update.notes:
        try:
            notes_frame = slide.notes_slide.notes_text_frame
            notes_frame.text = _trim_line(update.notes, max_len=400)
        except Exception:
            pass

    if update.remove_empty_placeholders:
        counters['removed_placeholders'] = _remove_empty_placeholders(slide)

    return counters


def apply_content_plan(input_pptx: Path, content_plan: ContentPlan, output_pptx: Path) -> dict[str, int]:
    prs = Presentation(str(input_pptx))
    totals = {'slides_updated': 0, 'removed_placeholders': 0}

    for update in content_plan.slides:
        if update.slide_index < 1 or update.slide_index > len(prs.slides):
            continue
        slide = prs.slides[update.slide_index - 1]
        counters = apply_slide_update(slide, update)
        totals['slides_updated'] += 1
        totals['removed_placeholders'] += counters['removed_placeholders']

    output_pptx.parent.mkdir(parents=True, exist_ok=True)
    prs.save(str(output_pptx))
    return totals


PLACEHOLDER_TOKENS = ['xxxx', 'lorem', 'ipsum', 'placeholder', 'todo', 'tbd']


def run_content_qa(pptx_path: Path) -> QAReport:
    text = _extract_text_markitdown(pptx_path)
    if not text:
        text = _extract_text_python_pptx(pptx_path)

    issues: list[QAIssue] = []
    current_slide = 1

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if line.lower().startswith('# slide '):
            token = line.replace('# Slide ', '').strip()
            if token.isdigit():
                current_slide = max(1, int(token))

        lowered = line.lower()
        for token in PLACEHOLDER_TOKENS:
            if token in lowered:
                issues.append(QAIssue(
                    slide_index=current_slide,
                    severity=QAIssueSeverity.high,
                    category=QAIssueCategory.placeholder,
                    description=f'Placeholder text detected: {line[:180]}',
                    fix_hint='Replace placeholder with final content',
                ))
                break

        if '\u2022' in line:
            issues.append(QAIssue(
                slide_index=current_slide,
                severity=QAIssueSeverity.medium,
                category=QAIssueCategory.other,
                description='Unicode bullet detected',
                fix_hint='Use native PowerPoint bullet paragraphs',
            ))

    prs = Presentation(str(pptx_path))
    for idx, slide in enumerate(prs.slides, start=1):
        has_text = any(shape.has_text_frame and shape.text.strip() for shape in slide.shapes)
        if not has_text:
            issues.append(QAIssue(
                slide_index=idx,
                severity=QAIssueSeverity.high,
                category=QAIssueCategory.other,
                description='Empty slide detected',
                fix_hint='Add content or remove slide',
            ))

    return QAReport(mode='content', issues=issues)


def _parse_slide_index_from_image(path: Path) -> int:
    digits = ''.join(ch for ch in path.stem if ch.isdigit())
    return int(digits) if digits.isdigit() else 1


def run_visual_qa_with_gemini(pptx_path: Path, output_dir: Path, gemini_api_key: str | None, model_name: str) -> QAReport:
    notes: list[str] = []

    if not gemini_api_key:
        notes.append('GEMINI_API_KEY missing, visual QA skipped')
        return QAReport(mode='visual', notes=notes)

    if _resolve_soffice_command() is None or not command_exists('pdftoppm'):
        notes.append('soffice or pdftoppm missing, visual QA skipped')
        return QAReport(mode='visual', notes=notes)

    images = render_slides_to_images(pptx_path, output_dir, prefix='qa-slide')
    notes.append(f'{len(images)} slide image(s) rendered for visual QA')

    genai.configure(api_key=gemini_api_key)
    model = genai.GenerativeModel(model_name)

    issues: list[QAIssue] = []
    for image_path in images:
        slide_idx = _parse_slide_index_from_image(image_path)
        prompt = (
            'Inspect this slide and return strict JSON: '
            '{"issues":[{"severity":"high|medium|low","category":"overflow|overlap|alignment|contrast|spacing|other","description":"...","fix_hint":"..."}]}. '
            'Return {"issues":[]} when no issue. '
            'Look for overlap, clipping, margins, contrast, alignment, spacing problems.'
        )

        with Image.open(image_path) as image:
            response = model.generate_content([prompt, image], generation_config={'temperature': 0.0})

        raw = getattr(response, 'text', '') or ''
        parsed = extract_json_object(raw)
        parsed_issues = parsed.get('issues') if isinstance(parsed, dict) else None

        if not isinstance(parsed_issues, list):
            lowered = raw.lower()
            if 'no issue' in lowered or '"issues":[]' in lowered or 'aucun' in lowered:
                continue
            issues.append(QAIssue(
                slide_index=slide_idx,
                severity=QAIssueSeverity.low,
                category=QAIssueCategory.other,
                description='Vision answer not parseable as JSON',
                fix_hint='Rerun visual QA',
            ))
            continue

        for item in parsed_issues:
            if not isinstance(item, dict):
                continue
            sev = str(item.get('severity', 'medium')).lower().strip()
            cat = str(item.get('category', 'other')).lower().strip()

            severity = {
                'high': QAIssueSeverity.high,
                'medium': QAIssueSeverity.medium,
                'low': QAIssueSeverity.low,
            }.get(sev, QAIssueSeverity.medium)

            category = {
                'overflow': QAIssueCategory.overflow,
                'overlap': QAIssueCategory.overlap,
                'alignment': QAIssueCategory.alignment,
                'contrast': QAIssueCategory.contrast,
                'spacing': QAIssueCategory.spacing,
                'placeholder': QAIssueCategory.placeholder,
                'typo': QAIssueCategory.typo,
                'other': QAIssueCategory.other,
            }.get(cat, QAIssueCategory.other)

            issues.append(QAIssue(
                slide_index=slide_idx,
                severity=severity,
                category=category,
                description=str(item.get('description', 'Visual issue'))[:300],
                fix_hint=str(item.get('fix_hint', 'Adjust layout'))[:300],
            ))

    return QAReport(mode='visual', issues=issues, notes=notes)


def merge_issue_counts(reports: list[QAReport]) -> int:
    return sum(len(report.issues) for report in reports)


SYSTEM_PROMPT_STRUCTURE = 'Return strict JSON for structural PPTX operations only.'
SYSTEM_PROMPT_CONTENT = 'Return strict JSON for slide content updates only.'
SYSTEM_PROMPT_FIX = 'Return strict JSON for content fixes based on QA issues.'


def _groq_model(settings: AgentSettings):
    model_name = settings.groq_model
    if model_name.startswith('groq:'):
        return model_name
    if settings.groq_api_key:
        provider = GroqProvider(api_key=settings.groq_api_key)
        return GroqModel(model_name, provider=provider)
    return f'groq:{model_name}'


def _analysis_json(analysis: DeckAnalysis) -> str:
    compact = {
        'source_path': analysis.source_path,
        'slide_count': analysis.slide_count,
        'detected_language': analysis.detected_language,
        'slides': [s.model_dump() for s in analysis.slides],
        'extracted_text': analysis.extracted_text[:8000],
    }
    return json.dumps(compact, ensure_ascii=False, indent=2)


def plan_structure(settings: AgentSettings, analysis: DeckAnalysis, user_instruction: str) -> StructurePlan:
    prompt = f'''
You are a PowerPoint structure planner.
Return only structural operations: delete_slide, duplicate_slide, add_layout_slide, reorder_slides.
Do not edit text content.
Use 1-based slide indexes.

User instruction:
{user_instruction}

Template analysis:
{_analysis_json(analysis)}
'''.strip()

    agent = Agent(
        _groq_model(settings),
        output_type=StructurePlan,
        model_settings=ModelSettings(temperature=0.1),
        instructions=SYSTEM_PROMPT_STRUCTURE,
    )

    try:
        return agent.run_sync(prompt).output
    except Exception:
        return StructurePlan(rationale='Fallback: no structural operations', operations=[])


def plan_content(settings: AgentSettings, analysis: DeckAnalysis, user_instruction: str, language: str) -> ContentPlan:
    prompt = f'''
You are a PowerPoint content writer.
Return only a ContentPlan JSON with slide-level updates.
Default language: {language}.
Use short titles and clear bullets.

User instruction:
{user_instruction}

Template analysis:
{_analysis_json(analysis)}
'''.strip()

    agent = Agent(
        _groq_model(settings),
        output_type=ContentPlan,
        model_settings=ModelSettings(temperature=0.2),
        instructions=SYSTEM_PROMPT_CONTENT,
    )

    try:
        output = agent.run_sync(prompt).output
        if not output.language:
            output.language = language
        return output
    except Exception:
        return ContentPlan(language=language, slides=[])


def plan_content_fixes(settings: AgentSettings, analysis: DeckAnalysis, user_instruction: str, language: str, reports: list[QAReport]) -> ContentPlan:
    issues_payload = [
        {
            'mode': report.mode,
            'issues': [issue.model_dump() for issue in report.issues],
            'notes': report.notes,
        }
        for report in reports
    ]

    prompt = f'''
You fix PowerPoint content after QA findings.
Return only ContentPlan JSON with necessary content fixes.
Keep language: {language}.
Prioritize high then medium severity.

Original instruction:
{user_instruction}

QA issues:
{json.dumps(issues_payload, ensure_ascii=False, indent=2)}

Template analysis:
{_analysis_json(analysis)}
'''.strip()

    agent = Agent(
        _groq_model(settings),
        output_type=ContentPlan,
        model_settings=ModelSettings(temperature=0.1),
        instructions=SYSTEM_PROMPT_FIX,
    )

    try:
        output = agent.run_sync(prompt).output
        if not output.language:
            output.language = language
        return output
    except Exception:
        return ContentPlan(language=language, slides=[])


def _count_major_issues(reports: list[QAReport]) -> int:
    return sum(1 for report in reports for issue in report.issues if issue.severity.value in {'high', 'medium'})


class PPTXEditingPipeline:
    def __init__(self, settings: AgentSettings):
        self.settings = settings

    def _new_run_dir(self) -> Path:
        now = datetime.now().strftime('%Y%m%d-%H%M%S')
        run_dir = self.settings.workdir / f'run-{now}'
        run_dir.mkdir(parents=True, exist_ok=True)
        return run_dir

    def _run_qa(self, pptx_path: Path, run_dir: Path, pass_index: int) -> list[QAReport]:
        reports: list[QAReport] = [run_content_qa(pptx_path)]
        if self.settings.enable_visual_qa:
            visual_dir = run_dir / f'qa-pass-{pass_index}'
            reports.append(run_visual_qa_with_gemini(
                pptx_path,
                output_dir=visual_dir,
                gemini_api_key=self.settings.gemini_api_key,
                model_name=self.settings.gemini_vision_model,
            ))
        return reports

    def run(self, input_pptx: Path, output_pptx: Path, instruction: str) -> RunReport:
        input_pptx = input_pptx.resolve()
        output_pptx = output_pptx.resolve()
        if not input_pptx.exists():
            raise FileNotFoundError(f'Input PPTX not found: {input_pptx}')

        run_dir = self._new_run_dir()
        analysis_dir = run_dir / 'analysis'
        analysis = analyze_template(input_pptx, analysis_dir, default_language=self.settings.default_language)

        structure_plan = plan_structure(self.settings, analysis, instruction)

        unpacked_dir = run_dir / 'unpacked'
        unpack_pptx(input_pptx, unpacked_dir)
        apply_structure_plan(unpacked_dir, structure_plan)

        structured_pptx = run_dir / 'structured.pptx'
        pack_pptx(unpacked_dir, structured_pptx)

        post_structure_analysis = analyze_template(structured_pptx, run_dir / 'analysis-post-structure', default_language=analysis.detected_language or self.settings.default_language)
        language = post_structure_analysis.detected_language or self.settings.default_language

        content_plan = plan_content(self.settings, post_structure_analysis, instruction, language=language)

        current_pptx = run_dir / 'edited.pptx'
        apply_content_plan(structured_pptx, content_plan, current_pptx)

        qa_reports: list[QAReport] = []
        latest_reports = self._run_qa(current_pptx, run_dir, pass_index=0)
        qa_reports.extend(latest_reports)

        major_issues = _count_major_issues(latest_reports)
        zero_defect_verified = False

        if major_issues == 0:
            verification_reports = self._run_qa(current_pptx, run_dir, pass_index=1)
            qa_reports.extend(verification_reports)
            latest_reports = verification_reports
            major_issues = _count_major_issues(verification_reports)
            zero_defect_verified = major_issues == 0

        for loop_index in range(1, self.settings.max_fix_loops + 1):
            if major_issues == 0:
                zero_defect_verified = True
                break

            fix_plan = plan_content_fixes(
                self.settings,
                post_structure_analysis,
                instruction,
                language=language,
                reports=latest_reports,
            )

            if not fix_plan.slides:
                break

            apply_content_plan(current_pptx, fix_plan, current_pptx)
            latest_reports = self._run_qa(current_pptx, run_dir, pass_index=loop_index + 1)
            qa_reports.extend(latest_reports)
            major_issues = _count_major_issues(latest_reports)

        if not zero_defect_verified:
            raise RuntimeError('QA gate failed: no zero-defect verification pass reached')

        final_unpack = run_dir / 'final-unpacked'
        unpack_pptx(current_pptx, final_unpack)
        clean_unreferenced_files(final_unpack)
        pack_pptx(final_unpack, output_pptx)

        rendered_paths = [str(p) for p in sorted(run_dir.rglob('qa-slide-*.png'))]

        report = RunReport(
            input_pptx=str(input_pptx),
            output_pptx=str(output_pptx),
            instruction=instruction,
            structure_plan=structure_plan,
            content_plan=content_plan,
            qa_reports=qa_reports,
            final_issue_count=merge_issue_counts(latest_reports),
            artifacts=RunArtifacts(
                workdir=str(run_dir),
                unpacked_dir=str(unpacked_dir),
                analysis_text_path=str(analysis_dir / 'analysis.txt'),
                rendered_image_paths=rendered_paths,
                report_json_path=str(run_dir / 'run-report.json'),
            ),
        )

        report_path = run_dir / 'run-report.json'
        report_path.write_text(report.model_dump_json(indent=2), encoding='utf-8')
        return report


def run_autonomous_agent(input_pptx: str | Path, output_pptx: str | Path, instruction: str, settings: AgentSettings | None = None) -> RunReport:
    cfg = settings or load_settings()
    pipeline = PPTXEditingPipeline(cfg)
    return pipeline.run(Path(input_pptx), Path(output_pptx), instruction)


print('Agent code loaded. System prompts are in: SYSTEM_PROMPT_STRUCTURE, SYSTEM_PROMPT_CONTENT, SYSTEM_PROMPT_FIX')

In [ ]:
# Optional: upload a template PPTX from your local machine
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))

In [ ]:
# 4) Run the agent (edit these variables)
INPUT_PPTX = '/content/template.pptx'
OUTPUT_PPTX = '/content/output.pptx'

USER_PROMPT = '''
Refais cette presentation en francais, ton executive,
messages clairs, structure narrative concise, et bullets actionnables.
'''.strip()

if not Path(INPUT_PPTX).exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_PPTX}')

if not os.getenv('GROQ_API_KEY'):
    raise RuntimeError('GROQ_API_KEY is required (Colab secret or env var)')

settings = load_settings()
print('Running with model:', settings.groq_model)

report = run_autonomous_agent(
    input_pptx=INPUT_PPTX,
    output_pptx=OUTPUT_PPTX,
    instruction=USER_PROMPT,
    settings=settings,
)

print('Done.')
print('Output PPTX:', report.output_pptx)
print('Final issue count:', report.final_issue_count)
print('Run report JSON:', report.artifacts.report_json_path)

In [ ]:
# Optional: inspect full report and download output
report.model_dump()

In [ ]:
if IN_COLAB and Path(OUTPUT_PPTX).exists():
    from google.colab import files
    files.download(OUTPUT_PPTX)